In [1]:
from catboost import CatBoostClassifier
from category_encoders import TargetEncoder
from pandas import read_csv

df = read_csv("ml-fundamentals-and-applications-2025-10/final_proj_data.csv")
df.head()

,Var1,Var2,Var3,Var4,Var5,Var6,Var7,Var8,Var9,Var10,...,Var222,Var223,Var224,Var225,Var226,Var227,Var228,Var229,Var230,y
0,NaN,NaN,NaN,NaN,NaN,812.0,14.0,NaN,NaN,NaN,...,catzS2D,jySVZNlOJy,NaN,xG3x,Aoh3,ZI9m,ib5G6X1eUxUn6,mj86,NaN,0
1,NaN,NaN,NaN,NaN,NaN,2688.0,7.0,NaN,NaN,NaN,...,i06ocsg,LM8l689qOp,NaN,kG3k,WqMG,RAYp,55YFVY9,mj86,NaN,0
2,NaN,NaN,NaN,NaN,NaN,1015.0,14.0,NaN,NaN,NaN,...,P6pu4Vl,LM8l689qOp,NaN,kG3k,Aoh3,ZI9m,R4y5gQQWY8OodqDV,am7c,NaN,0
3,NaN,NaN,NaN,NaN,NaN,168.0,0.0,NaN,NaN,NaN,...,BNrD3Yd,LM8l689qOp,NaN,NaN,FSa2,RAYp,F2FyR07IdsN7I,NaN,NaN,0
4,NaN,NaN,NaN,NaN,NaN,14.0,0.0,NaN,NaN,NaN,...,3B1QowC,LM8l689qOp,NaN,NaN,WqMG,RAYp,F2FyR07IdsN7I,NaN,NaN,0


In [2]:
from home_works.machine_learning.final_project.util import get_nan_percentage

allowed_columns = df.loc[:, get_nan_percentage(df) <= 50].columns.tolist()

df_cleaned = df[allowed_columns]
df_cleaned.head()

,Var6,Var7,Var13,Var21,Var22,Var24,Var25,Var28,Var35,Var38,...,Var218,Var219,Var220,Var221,Var222,Var223,Var226,Var227,Var228,y
0,812.0,14.0,1252.0,156.0,195.0,0.0,40.0,286.96,0.0,4850466.0,...,cJvF,AU8_WTd,4UxGlow,zCkv,catzS2D,jySVZNlOJy,Aoh3,ZI9m,ib5G6X1eUxUn6,0
1,2688.0,7.0,8820.0,364.0,455.0,4.0,288.0,200.00,0.0,132072.0,...,UYBR,AU8pNoi,GpvRJ5l,oslk,i06ocsg,LM8l689qOp,WqMG,RAYp,55YFVY9,0
2,1015.0,14.0,1784.0,136.0,170.0,2.0,40.0,294.48,0.0,3223524.0,...,cJvF,FzaX,ch2oGfM,zCkv,P6pu4Vl,LM8l689qOp,Aoh3,ZI9m,R4y5gQQWY8OodqDV,0
3,168.0,0.0,0.0,24.0,30.0,0.0,0.0,644.24,0.0,2135430.0,...,cJvF,FzaX,kH5mFX7,oslk,BNrD3Yd,LM8l689qOp,FSa2,RAYp,F2FyR07IdsN7I,0
4,14.0,0.0,0.0,36.0,45.0,0.0,0.0,239.84,0.0,3110400.0,...,cJvF,FzaX,x_lYlW4,oslk,3B1QowC,LM8l689qOp,WqMG,RAYp,F2FyR07IdsN7I,0


In [3]:
num_cols = df_cleaned.select_dtypes(include=["number"]).columns.tolist()
num_cols.remove("y")
num_cols

['Var6',
 'Var7',
 'Var13',
 'Var21',
 'Var22',
 'Var24',
 'Var25',
 'Var28',
 'Var35',
 'Var38',
 'Var44',
 'Var57',
 'Var65',
 'Var72',
 'Var73',
 'Var74',
 'Var76',
 'Var78',
 'Var81',
 'Var83',
 'Var85',
 'Var94',
 'Var109',
 'Var112',
 'Var113',
 'Var119',
 'Var123',
 'Var125',
 'Var126',
 'Var132',
 'Var133',
 'Var134',
 'Var140',
 'Var143',
 'Var144',
 'Var149',
 'Var153',
 'Var160',
 'Var163',
 'Var173',
 'Var181']

In [4]:
cat_cols = df_cleaned.drop(columns=num_cols).columns.tolist()
cat_cols.remove("y")
cat_cols

['Var192',
 'Var193',
 'Var195',
 'Var196',
 'Var197',
 'Var198',
 'Var199',
 'Var200',
 'Var202',
 'Var203',
 'Var204',
 'Var205',
 'Var206',
 'Var207',
 'Var208',
 'Var210',
 'Var211',
 'Var212',
 'Var214',
 'Var216',
 'Var217',
 'Var218',
 'Var219',
 'Var220',
 'Var221',
 'Var222',
 'Var223',
 'Var226',
 'Var227',
 'Var228']

In [5]:
from home_works.machine_learning.final_project.util import split_df_to_x_y

X, y = split_df_to_x_y(df_cleaned, "y")

In [6]:
from home_works.machine_learning.final_project.util import VarMeanEstimator
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

num_data_prep = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("var_mean", VarMeanEstimator(col_name="VarMean", cols_to_mean=num_cols))
])

num_data_prep.set_output(transform="pandas")

X_num = num_data_prep.fit_transform(X[num_cols])

X_num.head()

,Var6,Var7,Var13,Var21,Var22,Var24,Var25,Var28,Var35,Var38,...,Var140,Var143,Var144,Var149,Var153,Var160,Var163,Var173,Var181,VarMean
0,-0.212337,1.192359,0.017317,-0.142524,-0.140019,-0.488119,-0.263571,0.641706,-0.228852,0.817387,...,-0.291121,-0.085463,1.354143,-0.389478,1.040498,-0.086067,-0.553093,-0.046956,-0.225677,0.238947
1,0.617608,0.020968,2.832081,0.253591,0.255808,-0.024392,0.943577,-0.252876,-0.228852,-0.804485,...,-0.408379,-0.085463,-0.223836,0.216229,-0.984923,-0.198993,0.063617,-0.046956,-0.225677,0.081564
2,-0.122530,1.192359,0.215183,-0.180612,-0.178080,-0.256256,-0.263571,0.719067,-0.228852,0.258152,...,1.487849,-0.085463,1.354143,0.454210,1.007088,-0.131238,0.611261,-0.046956,-0.225677,0.228111
3,-0.497243,-1.150423,-0.448339,-0.393904,-0.391218,-0.488119,-0.458272,4.317150,-0.228852,-0.115863,...,-0.428480,-0.085463,-0.223836,-0.389478,1.202683,-0.379674,-0.553093,-0.046956,-0.225677,0.028790
4,-0.565373,-1.150423,-0.448339,-0.371052,-0.368381,-0.488119,-0.458272,0.156969,-0.228852,0.219268,...,-0.428480,-0.085463,-1.012826,-0.389478,-0.840864,-0.402259,-0.553093,-0.046956,-0.225677,-0.383421


In [7]:
from home_works.machine_learning.final_project.util import concat_dfs

X_final = concat_dfs(X_num, X[cat_cols])
X_final.head()

,Var6,Var7,Var13,Var21,Var22,Var24,Var25,Var28,Var35,Var38,...,Var217,Var218,Var219,Var220,Var221,Var222,Var223,Var226,Var227,Var228
0,-0.212337,1.192359,0.017317,-0.142524,-0.140019,-0.488119,-0.263571,0.641706,-0.228852,0.817387,...,G8WR,cJvF,AU8_WTd,4UxGlow,zCkv,catzS2D,jySVZNlOJy,Aoh3,ZI9m,ib5G6X1eUxUn6
1,0.617608,0.020968,2.832081,0.253591,0.255808,-0.024392,0.943577,-0.252876,-0.228852,-0.804485,...,5smi,UYBR,AU8pNoi,GpvRJ5l,oslk,i06ocsg,LM8l689qOp,WqMG,RAYp,55YFVY9
2,-0.122530,1.192359,0.215183,-0.180612,-0.178080,-0.256256,-0.263571,0.719067,-0.228852,0.258152,...,8m7I,cJvF,FzaX,ch2oGfM,zCkv,P6pu4Vl,LM8l689qOp,Aoh3,ZI9m,R4y5gQQWY8OodqDV
3,-0.497243,-1.150423,-0.448339,-0.393904,-0.391218,-0.488119,-0.458272,4.317150,-0.228852,-0.115863,...,d4ij,cJvF,FzaX,kH5mFX7,oslk,BNrD3Yd,LM8l689qOp,FSa2,RAYp,F2FyR07IdsN7I
4,-0.565373,-1.150423,-0.448339,-0.371052,-0.368381,-0.488119,-0.458272,0.156969,-0.228852,0.219268,...,NEOV,cJvF,FzaX,x_lYlW4,oslk,3B1QowC,LM8l689qOp,WqMG,RAYp,F2FyR07IdsN7I


In [8]:
from home_works.machine_learning.final_project.util import HyperParametersOptimizer

best_params = HyperParametersOptimizer(X_final, y, cat_cols=cat_cols).get_catboost_best_params(n_trials=100)

best_params

[I 2025-11-24 19:42:33,434] A new study created in memory with name: no-name-3bdd85a1-b67d-433d-b4b8-4e54278abf7c


0:	learn: 0.7801393	total: 63ms	remaining: 28.5s
50:	learn: 0.9792249	total: 362ms	remaining: 2.85s
100:	learn: 0.9996729	total: 641ms	remaining: 2.23s
150:	learn: 1.0000000	total: 943ms	remaining: 1.89s
200:	learn: 1.0000000	total: 1.25s	remaining: 1.57s
250:	learn: 1.0000000	total: 1.59s	remaining: 1.28s
300:	learn: 1.0000000	total: 1.87s	remaining: 944ms
350:	learn: 1.0000000	total: 2.14s	remaining: 622ms
400:	learn: 1.0000000	total: 2.41s	remaining: 312ms
450:	learn: 1.0000000	total: 2.71s	remaining: 12ms
452:	learn: 1.0000000	total: 2.73s	remaining: 0us
0:	learn: 0.7807919	total: 3.79ms	remaining: 1.71s
50:	learn: 0.9781109	total: 300ms	remaining: 2.36s
100:	learn: 0.9988963	total: 613ms	remaining: 2.14s
150:	learn: 1.0000000	total: 913ms	remaining: 1.83s
200:	learn: 1.0000000	total: 1.2s	remaining: 1.5s
250:	learn: 1.0000000	total: 1.49s	remaining: 1.2s
300:	learn: 1.0000000	total: 1.78s	remaining: 901ms
350:	learn: 1.0000000	total: 2.06s	remaining: 598ms
400:	learn: 1.0000000	to

[I 2025-11-24 19:42:49,373] Trial 0 finished with value: 0.6176156599889608 and parameters: {'weight_pos': 2.1216837525449996, 'iterations': 453, 'learning_rate': 0.2708003467071874, 'depth': 8, 'eval_metric': 'Accuracy', 'l2_leaf_reg': 0.0054820779948554325, 'bagging_temperature': 0.08865721759942813, 'rsm': 0.07541828191300892, 'min_data_in_leaf': 99, 'leaf_estimation_iterations': 8}. Best is trial 0 with value: 0.6176156599889608.


450:	learn: 1.0000000	total: 3.24s	remaining: 14.4ms
452:	learn: 1.0000000	total: 3.25s	remaining: 0us
0:	learn: 0.8719839	total: 4.31ms	remaining: 3.63s
50:	learn: 0.9051697	total: 1.02s	remaining: 15.9s
100:	learn: 0.9440828	total: 2.04s	remaining: 15s
150:	learn: 0.9467027	total: 3.01s	remaining: 13.8s
200:	learn: 0.9499643	total: 4.14s	remaining: 13.3s
250:	learn: 0.9527775	total: 5.43s	remaining: 12.9s
300:	learn: 0.9560846	total: 7.58s	remaining: 13.7s
350:	learn: 0.9584038	total: 9.82s	remaining: 13.8s
400:	learn: 0.9605298	total: 12.1s	remaining: 13.4s
450:	learn: 0.9622478	total: 13.4s	remaining: 11.7s
500:	learn: 0.9643523	total: 14.6s	remaining: 10s
550:	learn: 0.9669936	total: 16s	remaining: 8.53s
600:	learn: 0.9695491	total: 17.2s	remaining: 7s
650:	learn: 0.9719113	total: 18.5s	remaining: 5.52s
700:	learn: 0.9743165	total: 20.1s	remaining: 4.13s
750:	learn: 0.9764210	total: 21.5s	remaining: 2.69s
800:	learn: 0.9784396	total: 23.1s	remaining: 1.27s
844:	learn: 0.9800716	to

[I 2025-11-24 19:44:53,708] Trial 1 finished with value: 0.6036151063743931 and parameters: {'weight_pos': 37.94136822732985, 'iterations': 845, 'learning_rate': 0.014480453832860733, 'depth': 9, 'eval_metric': 'Accuracy', 'l2_leaf_reg': 0.0012287054579885955, 'bagging_temperature': 0.8874073819965811, 'rsm': 0.6208510822443597, 'min_data_in_leaf': 61, 'leaf_estimation_iterations': 2}. Best is trial 0 with value: 0.6176156599889608.


...

[I 2025-11-24 20:52:19,195] Trial 98 finished with value: 0.735779528418887 and parameters: {'weight_pos': 2.1783686296122036, 'iterations': 955, 'learning_rate': 0.05601813234905895, 'depth': 4, 'eval_metric': 'AUC', 'l2_leaf_reg': 27.81360857404398, 'bagging_temperature': 0.8307929633009974, 'rsm': 0.6100440113010361, 'min_data_in_leaf': 87, 'leaf_estimation_iterations': 9}. Best is trial 16 with value: 0.7404863910277388.


950:	total: 5.78s	remaining: 24.3ms
954:	total: 5.8s	remaining: 0us
0:	total: 7.87ms	remaining: 7.71s
50:	total: 367ms	remaining: 6.69s
100:	total: 727ms	remaining: 6.33s
150:	total: 1.09s	remaining: 6.02s
200:	total: 1.47s	remaining: 5.71s
250:	total: 1.86s	remaining: 5.42s
300:	total: 2.24s	remaining: 5.07s
350:	total: 2.63s	remaining: 4.71s
400:	total: 3.02s	remaining: 4.37s
450:	total: 3.41s	remaining: 4s
500:	total: 3.79s	remaining: 3.63s
550:	total: 4.17s	remaining: 3.25s
600:	total: 4.57s	remaining: 2.89s
650:	total: 4.95s	remaining: 2.51s
700:	total: 5.34s	remaining: 2.13s
750:	total: 5.73s	remaining: 1.75s
800:	total: 6.16s	remaining: 1.38s
850:	total: 6.54s	remaining: 1000ms
900:	total: 6.97s	remaining: 619ms
950:	total: 7.36s	remaining: 232ms
980:	total: 7.6s	remaining: 0us
0:	total: 7.79ms	remaining: 7.63s
50:	total: 375ms	remaining: 6.83s
100:	total: 739ms	remaining: 6.43s
150:	total: 1.1s	remaining: 6.04s
200:	total: 1.57s	remaining: 6.1s
250:	total: 1.97s	remaining: 5.73

[I 2025-11-24 20:53:00,863] Trial 99 finished with value: 0.7313288578880418 and parameters: {'weight_pos': 3.2330022549008297, 'iterations': 981, 'learning_rate': 0.058502394201555105, 'depth': 5, 'eval_metric': 'AUC', 'l2_leaf_reg': 23.10403362187623, 'bagging_temperature': 0.9132315226360548, 'rsm': 0.6058219303882414, 'min_data_in_leaf': 89, 'leaf_estimation_iterations': 9}. Best is trial 16 with value: 0.7404863910277388.


980:	total: 8.01s	remaining: 0us


{'weight_pos': 2.2251937977925387,
 'iterations': 990,
 'learning_rate': 0.043521575312755724,
 'depth': 5,
 'eval_metric': 'AUC',
 'l2_leaf_reg': 2.6361802237432617,
 'bagging_temperature': 0.9737877539995844,
 'rsm': 0.43468229033874983,
 'min_data_in_leaf': 31,
 'leaf_estimation_iterations': 3}

In [9]:
from home_works.machine_learning.final_project.util import split_train_test

X_train, X_test, y_train, y_test = split_train_test(X_final, y, balance_classes=False)

In [10]:
best_params

{'weight_pos': 2.2251937977925387,
 'iterations': 990,
 'learning_rate': 0.043521575312755724,
 'depth': 5,
 'eval_metric': 'AUC',
 'l2_leaf_reg': 2.6361802237432617,
 'bagging_temperature': 0.9737877539995844,
 'rsm': 0.43468229033874983,
 'min_data_in_leaf': 31,
 'leaf_estimation_iterations': 3}

In [12]:
from home_works.machine_learning.final_project.util import evaluate_model, split_train_test

best_params["class_weights"] = [1.0, best_params.pop("weight_pos")]
model = CatBoostClassifier(**best_params)

X_train, X_test, y_train, y_test = split_train_test(X_final, y)
evaluate_model(model, X_train, y_train, X_test, y_test, cat_cols=cat_cols)

ValueError: could not convert string to float: 'Qu0qrQKzJV'